In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# =====================================================================
# STEP 1: DEFINE A CUSTOM DATASET PIPELINE
# =====================================================================
class TabularDataset(Dataset):
    def __init__(self, features, labels):
        # Convert incoming data to standard PyTorch precision tensors
        self.X = torch.tensor(features, dtype=torch.float32)
        self.y = torch.tensor(labels, dtype=torch.float32).unsqueeze(1) # Align shape to (N, 1)
        
    def __len__(self):
        return len(self.X)
        
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Synthesize mock tabular tracking data (1000 records, 10 feature dimensions)
import numpy as np
np.random.seed(42)
raw_features = np.random.randn(1000, 10)
raw_labels = (np.sum(raw_features[:, :3], axis=1) > 0).astype(int) # Binary classification targets

# Initialize the Dataset instance
dataset = TabularDataset(raw_features, raw_labels)

# Instantiate the DataLoader
# shuffle=True mixes data indices every epoch to ensure smooth optimization pathways
data_loader = DataLoader(dataset, batch_size=32, shuffle=True, drop_last=False)

# =====================================================================
# STEP 2: BUILD A MODULAR NEURAL NETWORK CLASS
# =====================================================================
class DeepClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(DeepClassifier, self).__init__()
        
        # Build sequential block architecture
        self.feature_extractor = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU()
        )
        # Binary Classification output head (Yields raw logits)
        self.output_head = nn.Linear(hidden_dim // 2, 1)
        
    def forward(self, x):
        features = self.feature_extractor(x)
        logits = self.output_head(features)
        return logits

# Instantiate the model architecture
model = DeepClassifier(input_dim=10, hidden_dim=64)

# =====================================================================
# STEP 3: CONFIGURE OPTIMIZATION AND TRAINING RUNTIME
# =====================================================================
# nn.BCEWithLogitsLoss combines Sigmoid + Binary Cross-Entropy internally.
# It is more numerically stable than adding a Sigmoid layer explicitly to the model.
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

print("--- Initializing Execution Model Training Loop ---")
model.train() # Set model explicitly to training mode activation state

# Run a sample single epoch pass across the complete mini-batch stream
for epoch in range(1):
    epoch_loss = 0.0
    for batch_idx, (batch_X, batch_y) in enumerate(data_loader):
        # 1. Clear out gradient buffers from the previous iteration step
        optimizer.zero_grad()
        
        # 2. Forward Propagation (Compute raw logit predictions)
        predictions = model(batch_X)
        
        # 3. Quantify the mini-batch error metric
        loss = criterion(predictions, batch_y)
        
        # 4. Backpropagation (Calculate parameter derivative vectors)
        loss.backward()
        
        # 5. Gradient Descent (Update spatial weights and biases)
        optimizer.step()
        
        epoch_loss += loss.item()
        
    average_batch_loss = epoch_loss / len(data_loader)
    print(f"Epoch 1 Completed Success Track | Average Loss Per Mini-Batch: {average_batch_loss:.4f}")

--- Initializing Execution Model Training Loop ---
Epoch 1 Completed Success Track | Average Loss Per Mini-Batch: 0.3815
